In [23]:
import pandas as pd
import os

code_analysis_generated_path = "../../generated/code-analysis"
if not os.path.exists(code_analysis_generated_path):
    os.makedirs(code_analysis_generated_path)

reports_path = "../../generated/reports-no-bug"

data = []

users = os.listdir(reports_path)
if len(users) == 0:
    raise FileNotFoundError(f"Le dossier {reports_path} est vide. Copier les uploads dans ce dossier avant d'exécuter le notebook.")

for user in users:
    user_path = os.path.join(reports_path, user)
    jacoco_path = os.path.join(user_path, "JaCoCo", "jacoco.csv")
    pitest_path = os.path.join(user_path, "Pitest")
    jacoco_valid = os.path.exists(jacoco_path)
    data.append([user, jacoco_path, jacoco_valid])

main_df = pd.DataFrame(data, columns=['user', 'jacoco', 'jacoco_valid'])


# Add group

In [24]:
users_csv_path = '../../generated/database/users.csv'
if not os.path.exists(users_csv_path):
    raise FileNotFoundError(f"Le fichier {users_csv_path} n'existe pas. Exécutez le notebook 'notebooks/arrange data/Database data.ipynb' pour le générer.")

df_users = pd.read_csv(users_csv_path, usecols=['user', 'game_mode']).dropna()
main_df = main_df.merge(df_users, on='user', how='left')

In [25]:
main_df

,user,jacoco,jacoco_valid,game_mode
0,0947a22a-ec40-4f49-867b-330a623a1a35,../../generated/reports-no-bug\0947a22a-ec40-4...,True,TEAM
1,0d271530-be17-4538-bf04-dde3c6069b5f,../../generated/reports-no-bug\0d271530-be17-4...,True,SOLO
2,11555248-3f01-4d1c-9a71-ef7caf9150fa,../../generated/reports-no-bug\11555248-3f01-4...,False,SOLO
3,70c71a91-06a1-4bf4-8e56-f4327b1c8b3f,../../generated/reports-no-bug\70c71a91-06a1-4...,True,SOLO
4,70e85b5e-92d4-498a-9079-ba881f5b3b82,../../generated/reports-no-bug\70e85b5e-92d4-4...,True,TEAM
5,7234f6dc-a316-4576-8453-6e10d7cf1c3d,../../generated/reports-no-bug\7234f6dc-a316-4...,True,SOLO
6,84c2c4e6-1c27-4f2c-808b-9e84a7cb257e,../../generated/reports-no-bug\84c2c4e6-1c27-4...,True,SOLO
7,960ad9f8-2b83-4526-9cfb-1a59cec049a6,../../generated/reports-no-bug\960ad9f8-2b83-4...,False,SOLO
8,bcec21aa-2246-4ee8-9f2a-3998a5cde697,../../generated/reports-no-bug\bcec21aa-2246-4...,True,TEAM
9,c27240eb-52c0-4436-aa0f-e7f97a94a725,../../generated/reports-no-bug\c27240eb-52c0-4...,True,TEAM


# Get data where JaCoCo report is valid


In [26]:
cols = ['instruction', 'branch', 'line', 'method']
users_with_valid_jacoco = main_df[(main_df['jacoco_valid'] == True)]
jacoco_users = users_with_valid_jacoco[users_with_valid_jacoco['game_mode'].notna()]
print(f"Nombre d'utilisateurs avec un rapports JaCoCo valide : {len(jacoco_users)}")

Nombre d'utilisateurs avec un rapports JaCoCo valide : 11


In [27]:
jacoco_users

,user,jacoco,jacoco_valid,game_mode
0,0947a22a-ec40-4f49-867b-330a623a1a35,../../generated/reports-no-bug\0947a22a-ec40-4...,True,TEAM
1,0d271530-be17-4538-bf04-dde3c6069b5f,../../generated/reports-no-bug\0d271530-be17-4...,True,SOLO
3,70c71a91-06a1-4bf4-8e56-f4327b1c8b3f,../../generated/reports-no-bug\70c71a91-06a1-4...,True,SOLO
4,70e85b5e-92d4-498a-9079-ba881f5b3b82,../../generated/reports-no-bug\70e85b5e-92d4-4...,True,TEAM
5,7234f6dc-a316-4576-8453-6e10d7cf1c3d,../../generated/reports-no-bug\7234f6dc-a316-4...,True,SOLO
6,84c2c4e6-1c27-4f2c-808b-9e84a7cb257e,../../generated/reports-no-bug\84c2c4e6-1c27-4...,True,SOLO
8,bcec21aa-2246-4ee8-9f2a-3998a5cde697,../../generated/reports-no-bug\bcec21aa-2246-4...,True,TEAM
9,c27240eb-52c0-4436-aa0f-e7f97a94a725,../../generated/reports-no-bug\c27240eb-52c0-4...,True,TEAM
10,c27240eb-52c0-4436-aa0f-e7f97a94a725,../../generated/reports-no-bug\c27240eb-52c0-4...,True,SOLO
12,eda1acdb-c4bb-4425-a39f-4c25a998e739,../../generated/reports-no-bug\eda1acdb-c4bb-4...,True,SOLO


In [28]:
users_jacoco_data = []

for _, user_data in jacoco_users.iterrows():
    csv_file = pd.read_csv(user_data['jacoco']).drop(columns=['GROUP', 'PACKAGE', 'CLASS'])
    data = csv_file.sum().tolist()

    users_jacoco_data.append([user_data['user'], user_data['game_mode']] + data)

df_jacoco = pd.DataFrame(
    users_jacoco_data,
    columns=['user', 'game_mode', 'instruction_missed', 'instruction_covered',
             'branch_missed', 'branch_covered', 'line_missed', 'line_covered',
             'complexity_missed', 'complexity_covered', 'method_missed', 'method_covered']
)

for col in cols:
    df_jacoco[f'{col}'] = df_jacoco[f'{col}_covered'] / (df_jacoco[f'{col}_missed'] + df_jacoco[f'{col}_covered'])



df_jacoco.to_csv(f"{code_analysis_generated_path}/jacoco.csv", index=False)
df_jacoco


,user,game_mode,instruction_missed,instruction_covered,branch_missed,branch_covered,line_missed,line_covered,complexity_missed,complexity_covered,method_missed,method_covered,instruction,branch,line,method
0,0947a22a-ec40-4f49-867b-330a623a1a35,TEAM,3077,0,157,0,684,0,277,0,187,0,0.000000,0.000000,0.000000,0.000000
1,0d271530-be17-4538-bf04-dde3c6069b5f,SOLO,2817,260,143,14,630,54,249,28,167,20,0.084498,0.089172,0.078947,0.106952
2,70c71a91-06a1-4bf4-8e56-f4327b1c8b3f,SOLO,3005,72,157,0,666,18,269,8,179,8,0.023399,0.000000,0.026316,0.042781
3,70e85b5e-92d4-498a-9079-ba881f5b3b82,TEAM,3077,0,157,0,684,0,277,0,187,0,0.000000,0.000000,0.000000,0.000000
4,7234f6dc-a316-4576-8453-6e10d7cf1c3d,SOLO,3077,0,157,0,684,0,277,0,187,0,0.000000,0.000000,0.000000,0.000000
5,84c2c4e6-1c27-4f2c-808b-9e84a7cb257e,SOLO,3077,0,157,0,684,0,277,0,187,0,0.000000,0.000000,0.000000,0.000000
6,bcec21aa-2246-4ee8-9f2a-3998a5cde697,TEAM,2927,150,149,8,645,39,259,18,172,15,0.048749,0.050955,0.057018,0.080214
7,c27240eb-52c0-4436-aa0f-e7f97a94a725,TEAM,3077,0,157,0,684,0,277,0,187,0,0.000000,0.000000,0.000000,0.000000
8,c27240eb-52c0-4436-aa0f-e7f97a94a725,SOLO,3077,0,157,0,684,0,277,0,187,0,0.000000,0.000000,0.000000,0.000000
9,eda1acdb-c4bb-4425-a39f-4c25a998e739,SOLO,3077,0,157,0,684,0,277,0,187,0,0.000000,0.000000,0.000000,0.000000
